# Create Vector Representations of Independent Sources

This notebook accompanies "Figure 5: **Results: (mu)-ECoG Data**" of the manuscript, and produces the vector representations of the sources on the neural data application.

In [1]:
import torch
import numpy as np
import pandas as pd
import sys
sys.path.append("..")

from src.data import load_dataset
from src.sisr_algorithm import sisr
from src.models import SpectrogramMLP

## Compute Independent Components

In [2]:
dataset = "reach-2021-09-22"
task = "multioutput"
x_train, y_train, x_test, y_test, metadata = load_dataset(
    dataset=dataset, 
    task=task,
    data_path="../data"
)
tasks = ["classification", "classification"]
N, C, T = x_train.shape

# keep reach direction and reach time attributes
y_train = y_train[:, 2:4]
y_test = y_test[:, 2:4]

In [3]:
print(x_train.shape)
print(x_test.shape)

(394, 8, 2000)
(99, 8, 2000)


In [4]:
seeds = [0] # user can increase the number of seeds over which to take the maximum
W_init = None
lam = 1e-4
lr_unmix = 1e-3
lr_model = 1e-4
weight_decay = 0.01
density = "huber"
optim = "adam"
n_layers = 0

max_iter = 5000
eval_iter = 200
batch_size_trials = 32
batch_size_samples = 64
verbose = False

The training portion below can be skipped if the user wishes, as the output in `notebooks/output` will be used in the figure below.

In [5]:
best_acc = 0.0
best_W = None
best_models = None

for seed in seeds:
    models = [
        SpectrogramMLP(tasks[0], 1, 26, 81, n_classes=len(np.unique(y_train[:, 0])), n_layers=n_layers),
        SpectrogramMLP(tasks[1], 1, 26, 81, n_classes=len(np.unique(y_train[:, 1])), n_layers=n_layers),
    ]

    output = sisr(
        x_train, 
        density=density, 
        lr_unmix=lr_unmix, 
        lr_model=lr_model, 
        tasks=tasks,
        models=models,
        lam=lam,
        labels=y_train,
        batch_size_trials=batch_size_trials, 
        batch_size_samples=batch_size_samples, 
        weight_decay=weight_decay,
        seed=seed, 
        max_iter=max_iter,
        eval_iter=eval_iter,
        x_test=x_test,
        labels_test=y_test,
        mixing_mat=None,
        optim=optim,
        verbose=verbose,
        W_init=W_init
    )

    metrics = output["metrics"]
    print(f"seed {seed}:")
    print(f"\t Task 0 Accuracy: {metrics['test_accuracy_task_0'][-1]}")
    print(f"\t Task 1 Accuracy: {metrics['test_accuracy_task_1'][-1]}")

    if metrics['test_accuracy_task_0'][-1] > best_acc:
        best_acc = metrics['test_accuracy_task_0'][-1]
        best_W = output['unmixing_matrix']
        best_models = models

  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [01:02<00:00, 80.17it/s]


seed 0:
	 Task 0 Accuracy: 0.39393940567970276
	 Task 1 Accuracy: 0.8989899158477783


In [6]:
# save output, if needed. Models are used only for their extract features function, 
# np.save("output/figure_representations_best_W.npy", best_W)
torch.save(best_models[0].state_dict(), "output/figure_representations_best_model0.pt")
torch.save(best_models[1].state_dict(), "output/figure_representations_best_model1.pt")

## Extract Features

We use PCA (applied to standardized features) applied to the spectrogram of the given signals.

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multivariate_normal
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import NotFittedError

class StandardizedPCA:
    def __init__(self, n_components=None, standardize="whiten"):
        self.n_components = n_components
        self.standardize = standardize
        self.is_fitted = False

    def fit(self, x):
        if self.standardize == "whiten":
            self.scaler = StandardScaler().fit(x)
            self.pca = PCA(n_components=self.n_components).fit(self.scaler.transform(x))
        elif self.standardize == "normalize":
            z = x / np.linalg.norm(x, axis=1)[:, None]
            self.pca = PCA(n_components=self.n_components).fit(z)
        else:
            raise NotImplementedError
        self.is_fitted = True
        return self
    
    def __sklearn_is_fitted__(self):
        return self.is_fitted

    def transform(self, x):
        if not self.__sklearn_is_fitted__():
            return NotFittedError
        if self.standardize == "whiten":
            return self.pca.transform(self.scaler.transform(x))
        elif self.standardize == "normalize":
            z = x / np.linalg.norm(x, axis=1)[:, None]
            return self.pca.transform(z)
        else:
            raise NotImplementedError

    def fit_transform(self, x):
        return self.fit(x).transform(x)

In [8]:
# Compute full dimensionality features.

W = np.load("output/figure_representations_best_W.npy")
# W = output["unmixing_matrix"] # can also be taken from training run above.

s_tr = W @ x_train
s_te = W @ x_test

s0_tr = s_tr[:, 0, :] # reach direction
s1_tr = s_tr[:, 1, :] # stimulation protocol
s0_te = s_te[:, 0, :] # reach direction
s1_te = s_te[:, 1, :] # stimulation protocol

f0_tr = models[0].extract_features(torch.from_numpy(s0_tr)).numpy()
f1_tr = models[1].extract_features(torch.from_numpy(s1_tr)).numpy()
f0_te = models[0].extract_features(torch.from_numpy(s0_te)).numpy()
f1_te = models[1].extract_features(torch.from_numpy(s1_te)).numpy()

print(s_tr.shape)
print(s_te.shape)

print(s0_tr.shape)
print(s0_te.shape)
print(f0_tr.shape)
print(f1_te.shape)

(394, 8, 2000)
(99, 8, 2000)
(394, 2000)
(99, 2000)
(394, 2106)
(99, 2106)


Save output by running the cell below.

In [9]:
# spca0 = StandardizedPCA(n_components=2, standardize="whiten").fit(f0_tr)
# np.save("output/feat0_tr_whit.npy", spca0.transform(f0_tr))
# np.save("output/feat0_te_whit.npy", spca0.transform(f0_te))

# spca0 = StandardizedPCA(n_components=2, standardize="normalize").fit(f0_tr)
# np.save("output/feat0_tr_norm.npy", spca0.transform(f0_tr))
# np.save("output/feat0_te_norm.npy", spca0.transform(f0_te))

# spca1 = StandardizedPCA(n_components=2, standardize="whiten").fit(f1_tr)
# np.save("output/feat1_tr_whit.npy", spca1.transform(f1_tr))
# np.save("output/feat1_te_whit.npy", spca1.transform(f1_te))

# spca1 = StandardizedPCA(n_components=2, standardize="normalize").fit(f1_tr)
# np.save("output/feat1_tr_norm.npy", spca1.transform(f1_tr))
# np.save("output/feat1_te_norm.npy", spca1.transform(f1_te))